In [1]:
# !pip install tensorly
# !pip install tensorly-torch

In [2]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [3]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

n_epoch = 30

cuda


In [5]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 128

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [6]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [7]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [8]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [9]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [10]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8,256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [11]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))

append_to_file(file_name='TCL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [12]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [13]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [14]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')

append_to_file(file_name='TCL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TCL_report.txt', text=f'FC last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.014815330505371094, backward time : 0.003341197967529297
Train epoch 1: top1=0.5110399723052979%, top2=0.7085399627685547%, top3=0.812559962272644%, top4=0.8814199566841125%, top5=0.9251199960708618%, loss=0.01072003266453743, time=1.9940247535705566s
Test epoch 1: top1=0.6129999756813049%, top2=0.7979999780654907%, top3=0.8831999897956848%, top4=0.9341999888420105%, top5=0.9635999798774719%, loss=0.008568852120637894, time=0.3735044002532959s
Memory Usage  - Allocated: 32.59 MB, Reserved: 160.00 MB
forward time : 0.00018930435180664062, backward time : 0.00047659873962402344
Train epoch 2: top1=0.6552000045776367%, top2=0.825659990310669%, top3=0.9018399715423584%, top4=0.9444599747657776%, top5=0.9691799879074097%, loss=0.007683154176473617, time=1.502568244934082s
Test epoch 2: top1=0.6676999926567078%, top2=0.833899974822998%, top3=0.9068999886512756%, top4=0.946399986743927%, top5=0.9684999585151672%, loss=0.007539377272129059, time=0.32583

# TCL from Tensorly

In [15]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)
        self.tcl2 = TCL(input_shape = (64,2,2), rank = (10,1,1))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        # x = x.reshape(x.size(0), -1)  # Flatten
        # x = self.fc2(x)
        x = self.tcl2(x).squeeze()
        return x


model2 = CNN2().to(device)


In [16]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

In [17]:
classifier2 = nn.Sequential(
    TCL(input_shape = (64,8,8), rank = (64,2,2)),
    TCL(input_shape = (64,2,2), rank = (10,1,1))
)

print(cp(classifier2))
append_to_file(file_name='TCL_report.txt', text=f'TCL Tensorly classifier # parameters {cp(classifier2)}')

4772


In [18]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [19]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [20]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0008251667022705078, backward time : 0.0005414485931396484
Train epoch 1: top1=0.2569800019264221%, top2=0.4485599994659424%, top3=0.5900399684906006%, top4=0.6960399746894836%, top5=0.782039999961853%, loss=0.01555113543510437, time=1.656996488571167s
Test epoch 1: top1=0.37709999084472656%, top2=0.5916000008583069%, top3=0.7231999635696411%, top4=0.8155999779701233%, top5=0.8851000070571899%, loss=0.013399142479896545, time=0.3216841220855713s
Memory Usage  - Allocated: 24.80 MB, Reserved: 160.00 MB
forward time : 0.0007679462432861328, backward time : 0.0003650188446044922
Train epoch 2: top1=0.41923999786376953%, top2=0.6424999833106995%, top3=0.7710399627685547%, top4=0.8557800054550171%, top5=0.9081599712371826%, loss=0.012278728878498077, time=1.5407013893127441s
Test epoch 2: top1=0.45989999175071716%, top2=0.6855999827384949%, top3=0.8033999800682068%, top4=0.8786999583244324%, top5=0.9253999590873718%, loss=0.011581572139263153, time=0

In [21]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

# TCL new Method just 3D tensors

In [22]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False, device = device):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True)
          self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True)
          self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True)
          self.rank = rank
          # print(self.W.shape)
          self.device = device

    def forward(self, x):
          batch_size = x.shape[0]
          x = torch.matmul(x.view(batch_size, -1).to(device),torch.kron(torch.kron(self.w1,self.w2).to(device),self.w3).to(device))
      #     x = x.view((batch_size,) + self.rank)

          return x

In [23]:
# model = CNN3().to(device)
# for _, (inputs, targets) in enumerate(train_loader):
#         # inputs, targets = inputs.to(device), targets.to(device)
#         # output = model(inputs)
#         # print(f'{_} done')
#         if _ == 781:
#                 inputs, targets = inputs.to(device), targets.to(device)
#                 break

In [24]:
# inputs.shape

In [25]:
# conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1).to(device)
# conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1).to(device)
# pool = nn.MaxPool2d(2, 2).to(device)

# x1 = conv1(inputs)
# x2 = pool(x1)
# x3 = conv2(x2)
# x4 = pool(x3)
# x4.shape

In [26]:
# # temp = torch.rand((64,64,8,8)).to(device)
# tcl2 = TCL2(input_shape=(64,8,8), rank=(64,2,2)).to(device)
# temp2 = tcl2(x4)
# temp2.shape

In [27]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL2(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)
        self.tcl2 = TCL2(input_shape = (64,2,2), rank = (10,1,1))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        # x = x.reshape(x.size(0), -1)  # Flatten
        # x = self.fc2(x)
        x = self.tcl2(x).squeeze()
        return x


model3 = CNN3().to(device)


In [28]:
classifier3 = nn.Sequential(
    TCL2(input_shape = (64,8,8), rank = (64,2,2)),
    TCL2(input_shape = (64,2,2), rank = (10,1,1))
)

print(cp(classifier3))
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 classifier # parameters {cp(classifier3)}')

4772


In [29]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [30]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [31]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0002429485321044922, backward time : 0.0003581047058105469
Train epoch 1: top1=0.09755999594926834%, top2=0.18587999045848846%, top3=0.2847000062465668%, top4=0.38909998536109924%, top5=0.489579975605011%, loss=0.02113765257358551, time=1.5626769065856934s
Test epoch 1: top1=0.09649999439716339%, top2=0.18709999322891235%, top3=0.28790000081062317%, top4=0.3856000006198883%, top5=0.4887999892234802%, loss=0.018190350103378296, time=0.3356339931488037s
Memory Usage  - Allocated: 24.99 MB, Reserved: 162.00 MB
forward time : 0.00024509429931640625, backward time : 0.00034928321838378906
Train epoch 2: top1=0.09829999506473541%, top2=0.19696000218391418%, top3=0.2982400059700012%, top4=0.4022199809551239%, top5=0.5015400052070618%, loss=0.01797716625213623, time=1.5219275951385498s
Test epoch 2: top1=0.10649999976158142%, top2=0.2313999980688095%, top3=0.3319000005722046%, top4=0.42389997839927673%, top5=0.5246999859809875%, loss=0.01779285840988159

In [32]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

# Method 2

In [33]:
# class TCL3(nn.Module):
#     def __init__(self, input_shape, rank, bias = False, device = device):
#           super(TCL3, self).__init__()
#           #suppose it is 3d :
#           self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True).to(device)
#           self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True).to(device)
#           self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True).to(device)
#           self.W = torch.kron(torch.kron(self.w1.to(device),self.w2.to(device)).to(device),self.w3.to(device)).to(device)
#           self.rank = rank
#           # print(self.W.shape)
#           self.device = device

#     def forward(self, x):
#           batch_size = x.shape[0]
#           x = torch.matmul(x.view(batch_size, -1).to(device),self.W).to(device)
#           x = x.view((batch_size,) + self.rank).to(device)

#           return x

In [34]:
# class CNN4(nn.Module):
#     def __init__(self):
#         super(CNN4, self).__init__()
#         self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
#         self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
#         self.pool = nn.MaxPool2d(2, 2)
#         # shape here is 64,8,8
#         self.tcl = TCL3(input_shape = (64,8,8), rank = (64,2,2))
#         # self.fc1 = nn.Linear(64 * 8 * 8, 256)
#         self.fc2 = nn.Linear(256,10)

#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         # x = x.view(x.size(0), -1)  # Flatten
#         x = F.relu(self.tcl(x))
#         x = x.reshape(x.size(0), -1)  # Flatten
#         x = self.fc2(x)
#         return x


# model4 = CNN4().to(device)


In [35]:
# classifier4 = nn.Sequential(
#     TCL3(input_shape = (64,8,8), rank = (64,2,2)),
#     nn.Linear(256,10)
# )

# print(cp(classifier4))
# append_to_file(file_name='TCL_report.txt', text=f'TCL method 3 classifier # parameters {cp(classifier4)}')

In [36]:
# import torch.optim as optim
# import time

# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model4.parameters())

In [37]:
#  # Define train and test functions (use examples)
# def train_epoch(loader, epoch):
#     model4.train()

#     start_time = time.time()
#     running_loss = 0.0
#     correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
#     flag = 1

#     for _, (inputs, targets) in enumerate(loader):
#         inputs, targets = inputs.to(device), targets.to(device)

#         optimizer.zero_grad()
#         s = time.time()
#         outputs = model4(inputs)
#         loss = criterion(outputs, targets)
#         if flag:
#             forward_time = time.time() - s

#         s = time.time()
#         loss.backward(retain_graph=True)
#         if flag:
#           backward_time = time.time() - s
#         optimizer.step()

#         running_loss += loss.item()
#         accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
#         for k in accuracies:
#             correct[k] += accuracies[k]['correct']

#     elapsed_time = time.time() - start_time
#     top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
#     avg_loss = running_loss / len(loader.dataset)

#     report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
#     print(report_train)

#     return report_train

# def test_epoch(loader, epoch):
#     model4.eval()

#     start_time = time.time()
#     running_loss = 0.0
#     correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

#     for _, (inputs, targets) in enumerate(loader):
#         inputs, targets = inputs.to(device), targets.to(device)

#         outputs = model4(inputs)
#         loss = criterion(outputs, targets)

#         running_loss += loss.item()
#         accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
#         for k in accuracies:
#             correct[k] += accuracies[k]['correct']

#     elapsed_time = time.time() - start_time
#     top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
#     avg_loss = running_loss / len(loader.dataset)

#     report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
#     print(report_test)

#     return report_test

In [38]:
# # n_epoch = 10
# flag = 1
# print(f'Training for {len(range(n_epoch))} epochs\n')
# start_time = time.time()
# for epoch in range(1,n_epoch+1):
#     report_train = train_epoch(train_loader, epoch)
#     report_test = test_epoch(test_loader, epoch)
#     if flag:
#       string = print_gpu_memory_usage("Memory Usage ")
#       flag = 0
#     # report = report_train + '\n' + report_test + '\n\n'
#     # if epoch % 10 == 0:
#         # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
#         # torch.save(model1.state_dict(), model_path)
#     # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
#     #     f.write(report)
# end_time = time.time()
# print(f'\n\ntime take : {end_time - start_time}')
# append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 took {end_time - start_time} time')
# append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 had {string}')
# append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 last epoch result:\n{report_train}\n{report_test}')

In [39]:
# append_to_file(file_name='TCL_report.txt', text=f'########################################')

# Method 3

In [40]:
class TCL4(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL4, self).__init__()
          #suppose it is 3d :
          self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True)
          self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True)
          self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True)

    def forward(self, x):
          x = torch.tensordot(x, self.w1, dims=([1],[0]))
          x = torch.tensordot(x, self.w2, dims = ([1],[0]))
          x = torch.tensordot(x, self.w3, dims = ([1],[0]))

          return x

In [41]:
# import torch  

# # Initialize dimensions  
# B, C, H, W = 1, 4, 3, 5  # Example dimensions  
# c, h, w = 11, 12, 13  # Output dimensions after applying weights  
# w1 = torch.rand(C, c)  # Shape (C, c) i.e. (4, 11)  
# w2 = torch.rand(H, h)  # Shape (H, h) i.e. (3, 12)  
# w3 = torch.rand(W, w)  # Shape (W, w) i.e. (5, 13)  

# # Example input tensor of shape (B, C, H, W)  
# input_tensor = torch.rand(B, C, H, W)  

# # Step 1: Apply tensordot with w1 along the channel dimension  
# result1 = torch.tensordot(input_tensor, w1, dims=([1], [0]))  # shape = (B, H, W, c)  
# print(result1.shape)

# # Step 2: Apply tensordot with w2 along the height dimension  
# result2 = torch.tensordot(result1, w2, dims=([1], [0]))  # shape = (B, W, c, h)  
# print(result2.shape)

# # Step 3: Apply tensordot with w3 along the width dimension  
# final_result = torch.tensordot(result2, w3, dims=([1], [0]))  # shape = (B, c, h, w)  

# # Final shape  
# print("Final result shape:", final_result.shape)  # Should be (10, 11, 12, 13)

In [42]:
class CNN5(nn.Module):
    def __init__(self):
        super(CNN5, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL4(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # self.fc2 = nn.Linear(256,10)
        self.tcl2 = TCL4(input_shape = (64,2,2), rank = (10,1,1))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        # x = x.reshape(x.size(0), -1)  # Flatten
        # x = self.fc2(x)
        x = self.tcl2(x).squeeze()
        return x


model5 = CNN5().to(device)


In [43]:
classifier5 = nn.Sequential(
    TCL4(input_shape = (64,8,8), rank = (64,2,2)),
    TCL4(input_shape = (64,2,2), rank = (10,1,1))
)

print(cp(classifier5))
append_to_file(file_name='TCL_report.txt', text=f'TCL method 3 classifier # parameters {cp(classifier5)}')

4772


In [44]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model5.parameters())

In [45]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model5.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model5(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward(retain_graph=True)
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model5.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model5(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [46]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 3 took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 3 had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 3 last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0003490447998046875, backward time : 0.0004355907440185547
Train epoch 1: top1=0.09995999932289124%, top2=0.19993999600410461%, top3=0.2994000017642975%, top4=0.398499995470047%, top5=0.4967999756336212%, loss=0.023193131823539735, time=1.5011851787567139s
Test epoch 1: top1=0.10089999437332153%, top2=0.211899995803833%, top3=0.3010999858379364%, top4=0.4068000018596649%, top5=0.5065999627113342%, loss=0.017862181639671324, time=0.33408069610595703s
Memory Usage  - Allocated: 25.18 MB, Reserved: 162.00 MB
forward time : 0.00032448768615722656, backward time : 0.0004520416259765625
Train epoch 2: top1=0.10198000073432922%, top2=0.2124200016260147%, top3=0.3307799994945526%, top4=0.43299999833106995%, top5=0.5421599745750427%, loss=0.01727991614818573, time=1.514404058456421s
Test epoch 2: top1=0.10049999505281448%, top2=0.21119999885559082%, top3=0.33309999108314514%, top4=0.43539997935295105%, top5=0.548799991607666%, loss=0.017265044391155244, 

In [47]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')